# Multilingual Keyword-Based Retrieval (BM25) Implementation

Author: Daria

This notebook implements and tests the baseline retrieval component for the project assignment:
- Indexes documents in their original language (EN or DE)
- Detects query language
- Translates the query to the other language
- Searches both EN and DE documents using BM25 for cross-lingual retrieval


In [ ]:
!pip install deep-translator nltk rank-bm25 langdetect

In [22]:
import os
from typing import List, Dict, Any, Tuple, Optional
import numpy as np
from pathlib import Path
import json
import re

# For language detection
from langdetect import detect

# For translation
from deep_translator import GoogleTranslator

# For BM25 retrieval
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

## 1. Define the Multilingual BM25 Retriever Class

In [23]:
class MultilingualBM25Retriever:
    """
    A retrieval system that:
    1. Indexes documents in their original language (EN or DE)
    2. Detects query language
    3. Translates the query to the other language
    4. Searches both EN and DE documents using BM25
    """
    
    def __init__(self, docs_directory: str = "data/documents"):
        """
        Initialize the retriever.
        
        Args:
            docs_directory: Path to the directory containing JSON documents
        """
        self.docs_directory = docs_directory
        
        # Storage for documents by language
        self.documents = {
            "en": [],
            "de": []
        }
        
        # BM25 indexes for each language
        self.bm25_indexes = {
            "en": None,
            "de": None
        }
        
        # Tokenized corpus for each language
        self.tokenized_corpus = {
            "en": [],
            "de": []
        }
        
        # Load stopwords
        self.stopwords = {
            "en": set(stopwords.words('english')),
            "de": set(stopwords.words('german'))
        }
        
        # Load documents and build indexes
        self._load_documents()
        self._build_indexes()
    
    def _load_documents(self):
        """Load JSON documents from directory and separate by language"""
        print(f"Loading documents from {self.docs_directory}...")
        
        # Get all JSON files in the directory
        json_files = [f for f in os.listdir(self.docs_directory) if f.endswith('.json')]
        
        for file_name in json_files:
            file_path = os.path.join(self.docs_directory, file_name)
            
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    doc = json.load(f)
                    
                language = doc.get("language", "").lower()
                if language in ["en", "de"]:
                    self.documents[language].append(doc)
                else:
                    print(f"Skipping document with unknown language: {language} - {file_path}")
            
            except Exception as e:
                print(f"Error loading {file_path}: {e}")
        
        print(f"Loaded {len(self.documents['en'])} English documents and {len(self.documents['de'])} German documents")
    
    def _preprocess_text(self, text: str, language: str) -> List[str]:
        """
        Enhanced preprocessing with better handling of technical terms 
        
        Args:
            text: Text to preprocess
            language: Language of the text ('en' or 'de')
            
        Returns:
            List of tokens
        """
        if not text:
            return []
        
        # Normalize text 
        normalized_text = text.lower()
        
        # Special handling for technical terms that should be preserved intact
        # This helps with retrieval precision for technical documents
        technical_terms = []
        
        # Find technical terms like 'AI', 'KI', 'ETH', etc.
        tech_pattern = r'\b[A-Z]{2,}\b'  # Uppercase acronyms
        for match in re.finditer(tech_pattern, text):
            term = match.group().lower()
            technical_terms.append(term)
        
        # Find hyphenated technical terms
        hyphenated_words = re.findall(r'\w+-\w+', normalized_text)
        for hyphenated in hyphenated_words:
            # Add both the hyphenated version and the space-separated version
            normalized_text += " " + hyphenated.replace('-', ' ')
        
        # Tokenize
        tokens = word_tokenize(normalized_text)
        
        # Remove stopwords but keep alphanumeric tokens
        filtered_tokens = []
        for token in tokens:
            # Keep tokens that aren't pure stopwords
            if token not in self.stopwords.get(language, set()):
                # Keep alphanumeric tokens
                if any(c.isalnum() for c in token):
                    filtered_tokens.append(token)
        
        # Add the technical terms back to ensure they're included
        filtered_tokens.extend(technical_terms)
        
        return filtered_tokens
    
    def _build_indexes(self):
        """Build BM25 indexes with improved parameters for better relevance"""
        print("Building BM25 indexes...")
        
        # Process English documents
        for doc in self.documents["en"]:
            # Extract fields
            title = doc.get("title", "")
            content = doc.get("main_content", "")
            summary = doc.get("summary", "")
            keywords = doc.get("keywords", [])
            topics = doc.get("topics", [])
            
            # Improved weighting approach:
            # 1. Topic matching is critical, especially for technical subjects
            # 2. Title is very important for relevance
            # 3. Keywords are good signals
            # 4. Main content needs proper weight for longer documents
            weighted_parts = []
            
            # Add title with higher weight (6x)
            for _ in range(6):
                weighted_parts.append(title)
            
            # Topics have high weight (5x) - these help with topical relevance
            if topics:
                topics_text = " ".join(str(t) for t in topics)
                for _ in range(5):
                    weighted_parts.append(topics_text)
            
            # Keywords also have very high weight (5x)
            if keywords:
                keyword_text = " ".join(str(k) for k in keywords)
                for _ in range(5):
                    weighted_parts.append(keyword_text)
            
            # Add summary (3x)
            for _ in range(3):
                weighted_parts.append(summary)
            
            # Add content with proper weight (1x but it's typically longer)
            weighted_parts.append(content)
            
            # Join with spaces to create combined text
            combined_text = " ".join(weighted_parts)
            
            # Tokenize and add to corpus
            tokenized_doc = self._preprocess_text(combined_text, "en")
            self.tokenized_corpus["en"].append(tokenized_doc)
        
        # Process German documents with the same improved approach
        for doc in self.documents["de"]:
            title = doc.get("title", "")
            content = doc.get("main_content", "")
            summary = doc.get("summary", "")
            keywords = doc.get("keywords", [])
            topics = doc.get("topics", [])
            
            weighted_parts = []
            
            # Add title with high weight (6x)
            for _ in range(6):
                weighted_parts.append(title)
            
            # Topics have high weight (5x)
            if topics:
                topics_text = " ".join(str(t) for t in topics)
                for _ in range(5):
                    weighted_parts.append(topics_text)
            
            # Keywords also have very high weight (5x)
            if keywords:
                keyword_text = " ".join(str(k) for k in keywords)
                for _ in range(5):
                    weighted_parts.append(keyword_text)
            
            # Add summary (3x)
            for _ in range(3):
                weighted_parts.append(summary)
            
            # Add content
            weighted_parts.append(content)
            
            # Join with spaces
            combined_text = " ".join(weighted_parts)
            
            # Tokenize and add to corpus
            tokenized_doc = self._preprocess_text(combined_text, "de")
            self.tokenized_corpus["de"].append(tokenized_doc)
        
        # Use better BM25 parameters for small corpora
        # k1=2.0: Increase term frequency importance
        # b=0.25: Reduce document length normalization (works better for small corpora)
        if self.tokenized_corpus["en"]:
            print("Creating English BM25 index...")
            self.bm25_indexes["en"] = BM25Okapi(self.tokenized_corpus["en"], k1=2.0, b=0.25)
        
        if self.tokenized_corpus["de"]:
            print("Creating German BM25 index...")
            self.bm25_indexes["de"] = BM25Okapi(self.tokenized_corpus["de"], k1=2.0, b=0.25)
        
        print("BM25 indexes built successfully")
    
    def detect_language(self, query: str) -> str:
        """
        Detect the language of the query
        
        Args:
            query: User query
            
        Returns:
            Language code ('en' or 'de')
        """
        try:
            lang = detect(query)
            if lang == 'en':
                return 'en'
            elif lang == 'de':  # Only detect standard German
                return 'de'
            else:
                print(f"Language detected as {lang}, defaulting to English")
                return 'en'
        except:
            print("Language detection failed, defaulting to English")
            return 'en'
    
    def translate_query(self, query: str, source_lang: str, target_lang: str) -> str:
        """
        Translate query from source language to target language
        
        Args:
            query: Query to translate
            source_lang: Source language code ('en' or 'de')
            target_lang: Target language code ('en' or 'de')
            
        Returns:
            Translated query
        """
        try:
            translator = GoogleTranslator(source=source_lang, target=target_lang)
            translated = translator.translate(query)
            return translated
        except Exception as e:
            print(f"Translation error: {e}")
            return query  # Return original query if translation fails
    
    def search(self, query: str, top_k: int = 5) -> List[Dict]:
        """
        Improved search with smarter cross-lingual balancing
        
        Args:
            query: User query
            top_k: Number of top results to return
            
        Returns:
            List of retrieved documents with scores
        """
        # Detect query language
        query_lang = self.detect_language(query)
        
        # Get the other language
        other_lang = "de" if query_lang == "en" else "en"
        
        print(f"Query language detected as {query_lang}")
        
        # Translate query to the other language
        translated_query = self.translate_query(query, query_lang, other_lang)
        
        print(f"Original query: {query}")
        print(f"Translated query: {translated_query}")
        
        # Search in original language - get more results for better filtering
        original_results = self._search_language(query, query_lang, top_k * 2)
        
        # Search in translated language
        translated_results = self._search_language(translated_query, other_lang, top_k * 2)
        
        # More intelligent balancing:
        # 1. Favor highly-relevant primary language results (score > 0.5)
        # 2. Include top secondary language results for diversity
        # 3. Ensure relevant content (especially high-scored matches)
        
        # Combine all results
        all_results = original_results + translated_results
        
        # Sort by score (descending)
        all_results.sort(key=lambda x: x["score"], reverse=True)
        
        # Top half from primary language (if available)
        primary_results = [r for r in all_results if r["language"] == query_lang]
        # Include some high-scored results first
        high_scored_primary = [r for r in primary_results if r["score"] > 0]
        
        # Include some secondary language results
        secondary_results = [r for r in all_results if r["language"] == other_lang]
        # Include high-scored cross-lingual results
        high_scored_secondary = [r for r in secondary_results if r["score"] > 0]
        
        # Balanced allocation algorithm:
        # 1. Start with high-scoring results from both languages (if available)
        # 2. Fill the rest with best scoring results from either language
        balanced_results = []
        
        # Add high-scored primary language results first (up to 60% of total)
        primary_count = min(len(high_scored_primary), int(top_k * 0.6))
        balanced_results.extend(high_scored_primary[:primary_count])
        
        # Add high-scored secondary language results (up to 40% of total)
        secondary_count = min(len(high_scored_secondary), int(top_k * 0.4))
        balanced_results.extend(high_scored_secondary[:secondary_count])
        
        # Fill remaining spots with the best remaining results
        remaining_slots = top_k - len(balanced_results)
        if remaining_slots > 0:
            # Get results that weren't already included
            remaining_results = [r for r in all_results if r not in balanced_results]
            balanced_results.extend(remaining_results[:remaining_slots])
        
        # Final sort by score
        balanced_results.sort(key=lambda x: x["score"], reverse=True)
        
        return balanced_results[:top_k]
    
    def _search_language(self, query: str, language: str, top_k: int) -> List[Dict]:
        """
        Enhanced search with topic and keyword boosting
        
        Args:
            query: Query (in the language specified)
            language: Language to search in ('en' or 'de')
            top_k: Number of top results to return
            
        Returns:
            List of retrieved documents with scores
        """
        results = []
        
        # Check if we have documents and an index for this language
        if not self.documents[language] or self.bm25_indexes[language] is None:
            return results
        
        # Step 1: Run an initial search with the original query to identify relevant topics
        query_lower = query.lower()
        tokenized_query = self._preprocess_text(query, language)
        
        if not tokenized_query:
            return results
        
        # Get initial BM25 scores
        initial_scores = self.bm25_indexes[language].get_scores(tokenized_query)
        
        # Get initial top results to extract topics for query expansion
        initial_top_indices = np.argsort(initial_scores)[::-1][:3]  # Get top 3 initial matches
        
        # Step 2: Extract topics and keywords from initial top results to expand the query
        expansion_terms = set()
        
        for idx in initial_top_indices:
            if initial_scores[idx] > 0:  # Only use positively scored documents
                doc = self.documents[language][idx]
                
                # Extract key information from the document
                topics = doc.get("topics", [])
                keywords = doc.get("keywords", [])
                
                # Add topics and keywords to expansion terms
                for topic in topics:
                    # Add individual words from topics
                    for word in str(topic).lower().split():
                        if len(word) > 3 and word not in self.stopwords.get(language, set()):
                            expansion_terms.add(word)
                
                # Add keywords
                for keyword in keywords:
                    expansion_terms.add(str(keyword).lower())
        
        # Step 3: Create expanded query
        boosted_query = query
        
        # Add expansion terms if we found any
        if expansion_terms:
            boosted_query += " " + " ".join(expansion_terms)
        
        # Always add ETH to the query for institutional relevance
        if "eth" not in query_lower:
            boosted_query += " ETH"
        
        # Step 4: Run the final search with the expanded query
        final_tokenized_query = self._preprocess_text(boosted_query, language)
        
        if not final_tokenized_query:
            return results
        
        # Get BM25 scores with expanded query
        scores = self.bm25_indexes[language].get_scores(final_tokenized_query)
        
        # Get top_k document indices
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        # Post-score adjustment: Boost documents based on query-document relevance metrics
        adjusted_scores = []
        for idx in top_indices:
            score = float(scores[idx])
            doc = self.documents[language][idx]
            
            # Extract document metadata
            title = doc.get("title", "").lower()
            topics = [str(t).lower() for t in doc.get("topics", [])]
            
            # Boost documents with title and topic matches to original query
            for query_term in query_lower.split():
                # Skip very short terms
                if len(query_term) <= 2:
                    continue
                    
                # Title match is a strong signal
                if query_term in title:
                    score += 0.5
                
                # Topic match is also a good signal
                for topic in topics:
                    if query_term in topic:
                        score += 0.3
            
            adjusted_scores.append((idx, score))
        
        # Sort by adjusted score
        adjusted_scores.sort(key=lambda x: x[1], reverse=True)
        
        # Create result objects
        for idx, score in adjusted_scores:
            doc = self.documents[language][idx]
            
            result = {
                "document": doc,
                "score": score,
                "language": language,
                "title": doc.get("title", ""),
                "summary": doc.get("summary", ""),
                "content_snippet": self._get_content_snippet(doc.get("main_content", ""), final_tokenized_query)
            }
            
            results.append(result)
        
        return results[:top_k]
    
    def _get_content_snippet(self, content: str, query_tokens: List[str], max_length: int = 200) -> str:
        """
        Extract a relevant snippet from the content based on query tokens
        
        Args:
            content: Document content
            query_tokens: Tokenized query
            max_length: Maximum snippet length
            
        Returns:
            Content snippet
        """
        if not content or not query_tokens:
            return ""
        
        # Simple approach: Find first occurrence of any query token
        content_lower = content.lower()
        best_pos = len(content)
        
        for token in query_tokens:
            pos = content_lower.find(token)
            if pos != -1 and pos < best_pos:
                best_pos = pos
        
        # If no query tokens found, return the beginning of the content
        if best_pos == len(content):
            best_pos = 0
        
        # Find a good starting position (start of a sentence if possible)
        start = max(0, best_pos - 100)
        while start > 0 and content[start] not in ".!?\n":
            start -= 1
        
        if start > 0:
            start += 1  # Skip the punctuation
        
        # Find a good ending position (end of a sentence if possible)
        end = min(len(content), best_pos + max_length)
        while end < len(content) and content[end] not in ".!?\n":
            end += 1
        
        if end < len(content):
            end += 1  # Include the punctuation
        
        # Extract snippet
        snippet = content[start:end].strip()
        
        # Add ellipsis if needed
        if start > 0:
            snippet = "..." + snippet
        
        if end < len(content):
            snippet = snippet + "..."
        
        return snippet

## 2. Testing with Sample Documents

A small sample of documents to test the implementation.

In [24]:
import tempfile
import shutil

In [25]:
# Create temporary directory
temp_dir = tempfile.mkdtemp()
print(f"Created temporary directory: {temp_dir}")

Created temporary directory: /var/folders/97/_zj78ff90g9cg8jqzm6jx0cr0000gp/T/tmpql67oifl


In [26]:
# Sample documents
sample_docs = [
    {
        "language": "en",
        "title": "Artificial Intelligence Research at ETH",
        "date": "2023-01-15",
        "source": "ETH News",
        "main_content": "ETH Zurich is at the forefront of AI research in Europe. Researchers are working on machine learning, computer vision, and natural language processing. The university recently established a new AI center.",
        "summary": "ETH Zurich leads AI research with focus on machine learning and NLP.",
        "named_entities": ["ETH Zurich", "Europe", "AI center"],
        "topics": ["Artificial Intelligence", "Research", "Technology"],
        "keywords": ["AI", "machine learning", "research", "ETH"]
    },
    {
        "language": "en",
        "title": "Sustainable Energy Solutions",
        "date": "2022-11-05",
        "source": "ETH News",
        "main_content": "Researchers at ETH Zurich have developed new solar cell technology with improved efficiency. The cells convert more sunlight into electricity and can be manufactured at a lower cost. This could accelerate the adoption of renewable energy worldwide.",
        "summary": "ETH develops more efficient and cost-effective solar cells.",
        "named_entities": ["ETH Zurich"],
        "topics": ["Renewable Energy", "Sustainability", "Technology"],
        "keywords": ["solar cells", "renewable energy", "sustainability"]
    },
    {
        "language": "de",
        "title": "Künstliche Intelligenz Forschung an der ETH",
        "date": "2023-02-10",
        "source": "ETH Nachrichten",
        "main_content": "Die ETH Zürich ist führend in der KI-Forschung in Europa. Forscher arbeiten an maschinellem Lernen, Computer Vision und Verarbeitung natürlicher Sprache. Die Universität hat kürzlich ein neues KI-Zentrum eingerichtet.",
        "summary": "ETH Zürich führt KI-Forschung mit Fokus auf maschinelles Lernen und NLP.",
        "named_entities": ["ETH Zürich", "Europa", "KI-Zentrum"],
        "topics": ["Künstliche Intelligenz", "Forschung", "Technologie"],
        "keywords": ["KI", "maschinelles Lernen", "Forschung", "ETH"]
    },
    {
        "language": "de",
        "title": "Nachhaltige Energielösungen",
        "date": "2022-12-15",
        "source": "ETH Nachrichten",
        "main_content": "Forscher der ETH Zürich haben eine neue Solarzellentechnologie mit verbesserter Effizienz entwickelt. Die Zellen wandeln mehr Sonnenlicht in Elektrizität um und können zu geringeren Kosten hergestellt werden. Dies könnte die Einführung erneuerbarer Energien weltweit beschleunigen.",
        "summary": "ETH entwickelt effizientere und kostengünstigere Solarzellen.",
        "named_entities": ["ETH Zürich"],
        "topics": ["Erneuerbare Energie", "Nachhaltigkeit", "Technologie"],
        "keywords": ["Solarzellen", "erneuerbare Energie", "Nachhaltigkeit"]
    }
]

In [27]:
# Write sample documents to temporary directory
for i, doc in enumerate(sample_docs):
    file_path = os.path.join(temp_dir, f"doc_{i+1}.json")
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(doc, f, ensure_ascii=False, indent=2)

print(f"Created {len(sample_docs)} sample documents")

Created 4 sample documents


## 3. Initialize and Test the Retriever

In [28]:
import nltk

# Download required NLTK resources
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/donishchuk/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/donishchuk/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [29]:
# Initialize the retriever with our sample documents
retriever = MultilingualBM25Retriever(temp_dir)

Loading documents from /var/folders/97/_zj78ff90g9cg8jqzm6jx0cr0000gp/T/tmpql67oifl...
Loaded 2 English documents and 2 German documents
Building BM25 indexes...
Creating English BM25 index...
Creating German BM25 index...
BM25 indexes built successfully


In [30]:
# Test with English query
en_query = "What is ETH doing in artificial intelligence research?"
print("\n===== English Query =====")
results_en = retriever.search(en_query, top_k=3)

print(f"\nResults for query: '{en_query}'")
for i, result in enumerate(results_en):
    print(f"\n{i+1}. [{result['language'].upper()}] {result['title']} (Score: {result['score']:.4f})")
    print(f"Summary: {result['summary']}")
    print(f"Snippet: {result['content_snippet']}")


===== English Query =====
Query language detected as en
Original query: What is ETH doing in artificial intelligence research?
Translated query: Was macht ETH in der Forschung für künstliche Intelligenz?

Results for query: 'What is ETH doing in artificial intelligence research?'

1. [DE] Künstliche Intelligenz Forschung an der ETH (Score: 2.4360)
Summary: ETH Zürich führt KI-Forschung mit Fokus auf maschinelles Lernen und NLP.
Snippet: Die ETH Zürich ist führend in der KI-Forschung in Europa. Forscher arbeiten an maschinellem Lernen, Computer Vision und Verarbeitung natürlicher Sprache. Die Universität hat kürzlich ein neues KI-Zentrum eingerichtet.

2. [EN] Artificial Intelligence Research at ETH (Score: 1.8785)
Summary: ETH Zurich leads AI research with focus on machine learning and NLP.
Snippet: ETH Zurich is at the forefront of AI research in Europe. Researchers are working on machine learning, computer vision, and natural language processing. The university recently established 

In [31]:
# Test with German query
de_query = "Was macht die ETH in der KI-Forschung?"
print("\n===== German Query =====")
results_de = retriever.search(de_query, top_k=3)

print(f"\nResults for query: '{de_query}'")
for i, result in enumerate(results_de):
    print(f"\n{i+1}. [{result['language'].upper()}] {result['title']} (Score: {result['score']:.4f})")
    print(f"Summary: {result['summary']}")
    print(f"Snippet: {result['content_snippet']}")


===== German Query =====
Query language detected as de
Original query: Was macht die ETH in der KI-Forschung?
Translated query: What does ETH do in AI research?

Results for query: 'Was macht die ETH in der KI-Forschung?'

1. [DE] Künstliche Intelligenz Forschung an der ETH (Score: 0.8360)
Summary: ETH Zürich führt KI-Forschung mit Fokus auf maschinelles Lernen und NLP.
Snippet: Die ETH Zürich ist führend in der KI-Forschung in Europa. Forscher arbeiten an maschinellem Lernen, Computer Vision und Verarbeitung natürlicher Sprache. Die Universität hat kürzlich ein neues KI-Zentrum eingerichtet.

2. [EN] Artificial Intelligence Research at ETH (Score: 0.2785)
Summary: ETH Zurich leads AI research with focus on machine learning and NLP.
Snippet: ETH Zurich is at the forefront of AI research in Europe. Researchers are working on machine learning, computer vision, and natural language processing. The university recently established a new AI center.

3. [DE] Nachhaltige Energielösungen (Scor

## 4. Evaluate the Results

How well our BM25 retriever works across languages?

In [32]:
# Define some evaluation metrics

def print_retrieval_stats(query, results):
    """Print statistics about the retrieval results"""
    
    # Count results by language
    en_count = sum(1 for r in results if r['language'] == 'en')
    de_count = sum(1 for r in results if r['language'] == 'de')
    
    # Get original query language
    query_lang = retriever.detect_language(query)
    
    print(f"Query: '{query}'")
    print(f"Detected language: {query_lang.upper()}")
    print(f"Total results: {len(results)}")
    print(f"English results: {en_count}")
    print(f"German results: {de_count}")
    print(f"Cross-lingual results: {en_count if query_lang == 'de' else de_count}")
    
    # Print top result
    if results:
        top = results[0]
        print("\nTop result:")
        print(f"Title: {top['title']}")
        print(f"Language: {top['language'].upper()}")
        print(f"Score: {top['score']:.4f}")
    
    print("\n" + "-"*50)

In [33]:
# Test a few more queries
test_queries = [
    "solar energy research",
    "Solarenergieforschung",
    "machine learning at ETH",
    "maschinelles Lernen an der ETH",
    "sustainable technologies",
    "nachhaltige Technologien"
]

print("\n===== Evaluation =====")
for query in test_queries:
    results = retriever.search(query, top_k=3)
    print_retrieval_stats(query, results)


===== Evaluation =====
Query language detected as en
Original query: solar energy research
Translated query: Solarenergieforschung
Query: 'solar energy research'
Detected language: EN
Total results: 3
English results: 2
German results: 1
Cross-lingual results: 1

Top result:
Title: Sustainable Energy Solutions
Language: EN
Score: 0.6095

--------------------------------------------------
Query language detected as de
Original query: Solarenergieforschung
Translated query: Solar energy research
Query: 'Solarenergieforschung'
Detected language: DE
Total results: 3
English results: 2
German results: 1
Cross-lingual results: 2

Top result:
Title: Sustainable Energy Solutions
Language: EN
Score: 0.6095

--------------------------------------------------
Query language detected as en
Original query: machine learning at ETH
Translated query: maschinelles Lernen bei ETH
Query: 'machine learning at ETH'
Detected language: EN
Total results: 3
English results: 1
German results: 2
Cross-lingual r

In [34]:
import os
import json
import tempfile
import shutil
from pathlib import Path
from datetime import datetime
import re

def load_and_run_bm25_retriever():
    """
    Complete pipeline to:
    1. Load documents from HKNews structure
    2. Initialize the BM25 retriever
    3. Run queries and save results
    """
    # Step 1: Load documents from HKNews directory structure
    documents = load_hknews_documents()
    
    # Step 2: Store documents in a temporary directory for BM25 retriever
    temp_dir = create_temp_documents(documents)
    
    # Step 3: Initialize your MultilingualBM25Retriever with the documents
    retriever = MultilingualBM25Retriever(temp_dir)
    
    # Step 4: Run interactive search loop
    run_interactive_search(retriever)
    
    # Cleanup temp directory when done
    shutil.rmtree(temp_dir)
    print("Temporary files cleaned up")

def load_hknews_documents():
    """
    Load all JSON documents from the HKNews folder 
    """
    source_dir = Path("../HKNews/")
    documents = []
    
    print(f"Loading documents from {source_dir}...")
    
    # Check if directory exists
    if not source_dir.exists():
        print(f"Error: Directory {source_dir} not found")
        return documents
        
    # Get the first-level subdirectories (language folders)
    language_dirs = [d for d in source_dir.iterdir() if d.is_dir()]
    
    total_docs = 0
    en_docs = 0
    de_docs = 0
    
    # Process each language directory
    for lang_dir in language_dirs:
        # Extract language from directory name (de_internal → de)
        language = lang_dir.name.split('_')[0].lower()
        
        # Only process 'en' or 'de' languages
        if language not in ['en', 'de']:
            print(f"Skipping directory with unknown language prefix: {lang_dir.name}")
            continue
        
        # Process all year subdirectories
        year_dirs = [d for d in lang_dir.iterdir() if d.is_dir()]
        for year_dir in year_dirs:
            # Process all month subdirectories
            month_dirs = [d for d in year_dir.iterdir() if d.is_dir()]
            for month_dir in month_dirs:
                # Process all JSON files in this month
                json_files = [f for f in month_dir.iterdir() if f.suffix.lower() == '.json']
                
                for json_file in json_files:
                    try:
                        with open(json_file, 'r', encoding='utf-8') as f:
                            doc_data = json.load(f)
                        
                        # Extract required fields for the retriever
                        main_content = doc_data.get('main_content', '')
                        
                        # Skip documents with no content
                        if not main_content:
                            print(f"Skipping document with empty content: {json_file}")
                            continue
                        
                        # Create a unique document ID from the filename
                        doc_id = json_file.stem
                        
                        # Add required fields to the document
                        processed_doc = {
                            'id': doc_id,
                            'language': language,
                            'title': doc_data.get('title', ''),
                            'main_content': main_content,
                            'summary': doc_data.get('summary', ''),
                            'keywords': doc_data.get('keywords', []),
                            'topics': doc_data.get('topics', [])
                        }
                        
                        documents.append(processed_doc)
                        total_docs += 1
                        
                        if language == 'en':
                            en_docs += 1
                        else:
                            de_docs += 1
                        
                    except Exception as e:
                        print(f"Error loading {json_file}: {e}")
    
    print(f"Successfully loaded {total_docs} documents ({en_docs} English, {de_docs} German)")
    return documents

def create_temp_documents(documents):
    """
    Create a temporary directory with JSON documents for the retriever
    """
    # Create a temporary directory
    temp_dir = tempfile.mkdtemp()
    print(f"Creating temporary documents in {temp_dir}")
    
    # Write each document to a JSON file
    for i, doc in enumerate(documents):
        file_path = os.path.join(temp_dir, f"doc_{i+1}.json")
        with open(file_path, 'w', encoding='utf-8') as f:
            json.dump(doc, f, ensure_ascii=False, indent=2)
    
    print(f"Created {len(documents)} document files")
    return temp_dir

def save_results(query, results):
    """
    Save retrieval results to JSON files in the bm25_results directory
    """
    # Create results directory
    results_dir = Path("bm25_results")
    os.makedirs(results_dir, exist_ok=True)
    
    # Create a slug from the query for the filename
    slug = query.lower()
    slug = re.sub(r'\s+', '-', slug)  # Replace spaces with hyphens
    slug = re.sub(r'[^\w\-]', '', slug)  # Remove special characters
    slug = slug[:50]  # Limit length
    
    # Add timestamp to avoid overwriting
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"{slug}_{timestamp}.json"
    
    # Prepare the output data
    output_data = {
        "query": query,
        "timestamp": datetime.now().isoformat(),
        "results_count": len(results),
        "results": results
    }
    
    # Save to file
    output_path = results_dir / filename
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)
    
    print(f"Results saved to {output_path}")
    return output_path

def run_interactive_search(retriever):
    """
    Run an interactive search loop allowing the user to enter queries
    """
    print("\n=== BM25 Search Tool Ready ===")
    print("Enter your search queries below. Type 'exit' to quit.")
    
    while True:
        # Get query from user
        query = input("\nEnter search query: ")
        
        # Check if user wants to exit
        if query.lower() in ['exit', 'quit']:
            print("Exiting search tool")
            break
        
        # Skip empty queries
        if not query.strip():
            print("Please enter a valid query")
            continue
        
        # Ask for number of results
        try:
            k_str = input("Number of results to return (default: 5): ")
            top_k = int(k_str) if k_str.strip() else 5
        except ValueError:
            print("Invalid number, using default: 5")
            top_k = 5
        
        # Perform search
        results = retriever.search(query, top_k=top_k)
        
        # Display results
        print(f"\nFound {len(results)} results for '{query}':")
        for i, result in enumerate(results):
            print(f"\n{i+1}. [{result['language'].upper()}] {result['title']} (Score: {result['score']:.4f})")
            print(f"Summary: {result.get('summary', 'N/A')}")
            print(f"Snippet: {result.get('content_snippet', 'N/A')[:200]}...")
        
        # Ask if user wants to save results
        save_option = input("\nSave these results? (y/n): ")
        if save_option.lower() in ['y', 'yes']:
            save_results(query, results)

In [ ]:
load_and_run_bm25_retriever()

Loading documents from ../HKNews...
Skipping document with empty content: ../HKNews/en_internal/2022/10/should-eth-introduce-rules-for-meetings.json
Skipping document with empty content: ../HKNews/en_news_events/2022/11/protein-shapes-indicate-parkinsons-disease.json
Skipping document with empty content: ../HKNews/en_news_events/2022/11/art-created-by-computers.json
Skipping document with empty content: ../HKNews/en_news_events/2022/10/annette-oxenius-receives-the-cloetta-prize.json
Skipping document with empty content: ../HKNews/en_news_events/2022/10/trying-on-clothes-virtually.json
Skipping document with empty content: ../HKNews/en_news_events/2022/10/magma-on-mars-likely.json
Skipping document with empty content: ../HKNews/en_news_events/2022/10/fighting-tumours-with-magnetic-bacteria.json
Skipping document with empty content: ../HKNews/en_news_events/2022/09/the-e-sling-electric-aircraft-takes-off.json
Skipping document with empty content: ../HKNews/en_news_events/2024/08/millions


Enter search query:  what news about energy?
Number of results to return (default: 5):  5


Query language detected as en
Original query: what news about energy?
Translated query: Welche Nachrichten über Energie?

Found 5 results for 'what news about energy?':

1. [EN] (Photograph: ETH Zurich) (Score: 141.4725)
Summary: 12 EU-Commission: A hydrogen strategy for a climate-neutral Europe

13 Energate Messenger: Berner Regierungsrat will Aufbau von H2-Tankstellennetz unterstützen

14 Kanton Zurich Press Release: Regierungsrat künftig auch mit Wasserstoff-Fahrzeugen unterwegs

15 Corporate Europe: The Hydrogen Hype

16 See previous reference

17 Recharge news: Liebreich: ‘Oil sector is lobbying for inefficient hydrogen cars because it wants to delay electrification’

18 SRF: Open Letter to Prime Minister Boris Johnson about Hydrogen Strategy

About the author: Anthony Patt Professor of climate policy at ETH Zurich...
Snippet: Main article: (Photograph: ETH Zurich)

To save the climate, the world needs to stop using fossil fuels by mid-century. We are finally headed in the right d


Save these results? (y/n):  y


Results saved to bm25_results/what-news-about-energy_20250510_134121.json
